---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  Edit paths, hyperparameter defaults, and corpus flags here. ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, hashlib
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/Capstone/project"
RUNS_ROOT  = f"{DRIVE_ROOT}/models"
LABELS_ROOT  = f"{DRIVE_ROOT}/labels"

# Corpus label roots — add new corpora here
BATCHED_ROOT_LIBRI_100 = f"{LABELS_ROOT}/clean-100"
BATCHED_ROOT_LIBRI_360 = f"{LABELS_ROOT}/clean-360"
BATCHED_ROOT_SBC       = f"{LABELS_ROOT}/sbcsae"
BATCHED_ROOT_BU        = f"{LABELS_ROOT}/bu"
BATCHED_ROOT_PS        = f"{LABELS_ROOT}/ps"
# BATCHED_ROOT_NEWCORPUS = f"{DRIVE_ROOT}/labels/newcorpus"

# ── Corpus load flags ─────────────────────────────────────────────────────────
# Set False to skip loading a corpus (saves time if you won't use it this session).
# A run that requests a skipped corpus will raise a KeyError.
LOAD_LIBRI = True   # ~145k samples — slow to load
LOAD_SBC   = True   # ~4k samples
LOAD_BU    = True   # ~426 samples
LOAD_PS    = True   # People's Speech silver-standard
# LOAD_NEWCORPUS = True

# SBC test holdout IDs — these are NEVER added to the registry or any training set
SBC_TEST_IDS = set(f"SBC{str(n).zfill(3)}" for n in range(1, 6))  # SBC001–SBC005

# ── Data split (train/val only — test is always SBC001–005) ───────────────────
SPLIT_SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

# ── Text pre-processing ───────────────────────────────────────────────────────
STRIP_PUNCTUATION = False
EXTRA_FEATURES    = []

# ── POS feature flags (defaults — can be overridden per-run in Cell 14) ───────
# TRAIN_ON_TEXT : DistilBERT receives the real word sequence as input.
# TRAIN_ON_POS  : POS embeddings are injected post-transformer (combined mode).
#                 Requires TRAIN_ON_TEXT=True. POS-only mode (TRAIN_ON_TEXT=False,
#                 TRAIN_ON_POS=True) replaces input words with POS abbreviations.
# Neither True  : invalid — raises an error in run_experiment().
# Default here is text-only; override per run in the RUNS queue (Cell 14).
TRAIN_ON_TEXT     = True
TRAIN_ON_POS      = False
POS_EMBEDDING_DIM = 64      # embedding size before Linear projection to H=768
                             # only used in combined mode (text+POS)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3
WARMUP_STEPS  = 0
WEIGHT_DECAY  = 0.01

# ── Class-imbalance strategy (boundary head only) ─────────────────────────────
IMBALANCE_STRATEGY   = "none"
BOUNDARY_LOSS_WEIGHT = 5.0

# ── Multi-task loss weights ───────────────────────────────────────────────────
BOUNDARY_TASK_WEIGHT   = 1.0
INTONATION_TASK_WEIGHT = 1.0
BREAK_IDX_TASK_WEIGHT  = 1.0

# ── Run metadata ──────────────────────────────────────────────────────────────
RUN_NOTES = ""

# ── Dual checkpoint (optional) ────────────────────────────────────────────────
# When True, also saves the checkpoint with best val intonation F1 and runs a
# second test evaluation pass. Off by default — enable per-run via overrides.
SAVE_INTONATION_CHECKPOINT = True

# ── Model registry ─────────────────────────────────────────────────────────
# Tracks every run's deterministic name → parameters. Lives at RUNS_ROOT
# (parent of full/ and partial/), keyed by auto-incrementing model number.
MODEL_REGISTRY_PATH = f"{RUNS_ROOT}/model_registry.json"

print("✓ Configuration loaded.")
print(f"  Corpus flags:  LIBRI={LOAD_LIBRI}  SBC={LOAD_SBC}  BU={LOAD_BU}  PS={LOAD_PS}")
print(f"  POS flags:     TRAIN_ON_TEXT={TRAIN_ON_TEXT}  TRAIN_ON_POS={TRAIN_ON_POS}  POS_EMB_DIM={POS_EMBEDDING_DIM}")
print(f"  SBC test IDs:  {sorted(SBC_TEST_IDS)}")
print(f"  Dual checkpoint: {SAVE_INTONATION_CHECKPOINT}")

from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)
os.makedirs(RUNS_ROOT, exist_ok=True)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Deterministic run naming                                    ║
# ║  Computes a canonical name from dataset list + key params.   ║
# ║  Same inputs always produce the same name — no manual typing.║
# ╚══════════════════════════════════════════════════════════════╗

_DATASET_LETTERS = {"libri": "L", "ps": "P", "sbc": "S"}
_FULL_SET = {"libri", "ps", "sbc"}

_DEFAULT_LR      = 2e-5
_DEFAULT_WARMUP  = 0

def _format_lr(lr):
    """2e-05 → 'lr2e5'   1e-05 → 'lr1e5'   3e-05 → 'lr3e5'"""
    s = f"{lr:.0e}"            # e.g. "2e-05"
    mantissa, exp = s.split("e")
    exp_num = exp.lstrip("+-").lstrip("0") or "0"
    return f"lr{mantissa}e{exp_num}"

def compute_run_name(datasets: list[str], run_cfg: dict) -> tuple[str, str]:
    # ── Dataset letters, canonical order ───────────────────────────────────
    letters = "".join(
        _DATASET_LETTERS[d] for d in ["libri", "ps", "sbc"] if d in datasets
    )
    # ── Loss suffix (always present) ────────────────────────────────────────
    if run_cfg["IMBALANCE_STRATEGY"].startswith("weight"):
        _blw = run_cfg["BOUNDARY_LOSS_WEIGHT"]
        _blw_str = str(int(_blw)) if _blw == int(_blw) else str(_blw).replace(".", "p")
        loss_suffix = f"wl{_blw_str}"
    else:
        loss_suffix = "stl"
    parts = [letters, loss_suffix]

    # ── POS suffix (optional) ───────────────────────────────────────────────
    train_text = run_cfg["TRAIN_ON_TEXT"]
    train_pos  = run_cfg["TRAIN_ON_POS"]
    if train_pos and not train_text:
        parts.append("posonly")
    elif train_pos and train_text:
        parts.append("pos")
    # else: text-only, default, no suffix

    # ── Punctuation suffix (optional) ───────────────────────────────────────
    if not run_cfg["STRIP_PUNCTUATION"]:
        parts.append("pp")
    # else: stripped, default, no suffix

    # ── Learning rate suffix (optional) ─────────────────────────────────────
    if run_cfg["LEARNING_RATE"] != _DEFAULT_LR:
        parts.append(_format_lr(run_cfg["LEARNING_RATE"]))

    # ── Warmup suffix (optional) ────────────────────────────────────────────
    if run_cfg["WARMUP_STEPS"] != _DEFAULT_WARMUP:
        parts.append(f"warmup{run_cfg['WARMUP_STEPS']}")

    name = "_".join(parts)
    category = "full" if set(datasets) == _FULL_SET else "partial"
    return name, category

print("✓ compute_run_name() defined.")

def resolve_run_path(name: str, category: str) -> tuple[str, str]:
    """
    Resolve the final run directory for a given deterministic name,
    appending a numeric disambiguator (_2, _3, ...) if the name is
    already taken on disk. Never overwrites an existing run directory.
    """
    base_dir = os.path.join(RUNS_ROOT, category)
    os.makedirs(base_dir, exist_ok=True)

    candidate = name
    suffix = 1
    while os.path.isdir(os.path.join(base_dir, candidate)):
        suffix += 1
        candidate = f"{name}_{suffix}"

    run_dir = os.path.join(base_dir, candidate)
    return candidate, run_dir

def _load_registry() -> dict:
    """Load model_registry.json, or return an empty dict if it doesn't exist yet."""
    if os.path.exists(MODEL_REGISTRY_PATH):
        with open(MODEL_REGISTRY_PATH) as f:
            return json.load(f)
    return {}


def _next_model_number(registry: dict) -> int:
    """Next available model number. Starts at 1 if registry is empty."""
    if not registry:
        return 1
    return max(int(k) for k in registry) + 1


def register_model(name: str, run_dir: str, datasets: list[str], run_cfg: dict,
                   loss_suffix_label: str) -> int:
    """
    Append a new entry to model_registry.json and return its assigned number.

    Parameters
    ----------
    name               : deterministic run name, e.g. "LPS_stl" or "LS_wl3"
    run_dir             : full path to the run directory
    datasets            : list[str] used for training
    run_cfg             : resolved config dict (after overrides applied)
    loss_suffix_label   : "standard" or "weighted" — spelled out for the registry
                          (the folder/name suffix abbreviates to "stl", or to
                          "wl{boundary_loss_weight}" e.g. "wl3"/"wl5"/"wl7" —
                          the weight value is load-bearing in the name now,
                          since different weights produce materially
                          different results, not just a cosmetic variant)
    """
    registry = _load_registry()
    model_num = _next_model_number(registry)

    rel_path = os.path.relpath(run_dir, RUNS_ROOT)

    registry[str(model_num)] = {
        "name": name,
        "path": rel_path,
        "datasets": datasets,
        "loss": loss_suffix_label,
        "boundary_loss_weight": run_cfg["BOUNDARY_LOSS_WEIGHT"] if loss_suffix_label == "weighted" else None,
        "train_on_text": run_cfg["TRAIN_ON_TEXT"],
        "train_on_pos": run_cfg["TRAIN_ON_POS"],
        "strip_punctuation": run_cfg["STRIP_PUNCTUATION"],
        "learning_rate": run_cfg["LEARNING_RATE"],
        "warmup_steps": run_cfg["WARMUP_STEPS"],
        "num_epochs": run_cfg["NUM_EPOCHS"],
        "run_notes": run_cfg["RUN_NOTES"],
    }

    with open(MODEL_REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)

    return model_num

print("✓ resolve_run_path() defined.")
print("✓ Model registry functions defined.")


---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install packages                                   ║
# ╚══════════════════════════════════════════════════════════════╝

!pip install -q transformers datasets scikit-learn matplotlib

import torch
print(f"\n✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠  No GPU — training will be slow.")
    print("   Runtime → Change runtime type → T4 GPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")


---
## Section 3 · Imports

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, string, random, shutil, traceback, gc, hashlib, subprocess
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    get_linear_schedule_with_warmup,
)
from tqdm.notebook import tqdm, tqdm as tqdm_nb

class _NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)

print("✓ Imports complete.")


---
## Section 4 · Load Corpora

Loads each corpus based on the flags in Cell 1. Splits SBC into train (`sbc`) and fixed test holdout (`samples_sbc_test`) at load time.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load corpora                                       ║
# ║                                                              ║
# ║  Loads each corpus based on flags set in Cell 1.             ║
# ║  Builds CORPUS_REGISTRY (training data only) and             ║
# ║  fixed evaluation sets (never in registry):                  ║
# ║    samples_sbc_test — boundary + intonation eval (SBC001–005)║
# ║    samples_bu       — break index eval (full BU corpus)      ║
# ║                                                              ║
# ║  To add a future corpus:                                     ║
# ║    1. Add LOAD_NEWCORPUS flag + BATCHED_ROOT_NEWCORPUS in C1 ║
# ║    2. Add a load block below (copy any existing block)       ║
# ║    3. Add key → samples to CORPUS_REGISTRY at the bottom     ║
# ╚══════════════════════════════════════════════════════════════╝

def load_label_files_batched(batched_root, label=""):
    """
    Load all samples from batch_*.json files in batched_root.

    Batch file layout (produced by annotation_pipeline_*.ipynb):
        batch_0000.json  ← dict keyed by sample_id, each value has "b", "i", "x"
        ...

    Returns
    -------
    samples : list[dict]  — keys: sample_id, tokens, boundary_labels,
                            intonation_labels, break_idx_labels
    meta    : dict        — empty placeholder (meta.json not bundled)
    """
    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        raise FileNotFoundError(
            f"No batch files found in {batched_root}.\n"
            "Check path or run the relevant annotation pipeline first."
        )
    tag = f" [{label}]" if label else ""
    print(f"Found {len(batch_files)} batch files in {batched_root}{tag}.")

    samples, n_skipped = [], 0
    for fname in tqdm(batch_files, desc=f"Loading{tag}", unit="batch"):
        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)
            for sid, data in batch.items():
                b, i, x = data["b"], data["i"], data["x"]
                n = len(b["tokens"])
                x_labels = x["labels"] if x is not None else [""] * n
                if not (len(b["consensus"]) == len(i["labels"]) == len(x_labels) == n):
                    print(f"  ⚠ {sid}: mismatched label lengths — skipping.")
                    n_skipped += 1
                    continue
                samples.append({
                    "sample_id":         sid,
                    "tokens":            b["tokens"],
                    "boundary_labels":   b["consensus"],
                    "intonation_labels": i["labels"],
                    "break_idx_labels":  x_labels,
                })

    print(f"  ✓ {len(samples):,} samples loaded  ({n_skipped} skipped).")
    if samples:
        total_tokens     = sum(len(s["tokens"])          for s in samples)
        total_boundaries = sum(sum(s["boundary_labels"]) for s in samples)
        pos_rate         = total_boundaries / max(total_tokens, 1)
        ratio            = (1 - pos_rate) / max(pos_rate, 1e-9)
        print(f"  Total words:     {total_tokens:,}")
        print(f"  Boundary rate:   {100*pos_rate:.1f}%  (ratio ≈ {ratio:.1f}:1)")
        print(f"  → Suggested BOUNDARY_LOSS_WEIGHT: ~{round(ratio)}.0")
    return samples, {}


# ── CORPUS_REGISTRY (training data only) ─────────────────────────────────────
# Maps name → sample list. Only loaded corpora appear here.
# Valid keys: "libri", "sbc", "ps"
# BU is evaluation-only and never enters the registry — see below.
CORPUS_REGISTRY = {}

# ── LibriTTS ──────────────────────────────────────────────────────────────────
if LOAD_LIBRI:
    _samples_100, _ = load_label_files_batched(BATCHED_ROOT_LIBRI_100, "libri-100")
    _samples_360, _ = load_label_files_batched(BATCHED_ROOT_LIBRI_360, "libri-360")
    samples_libri = _samples_100 + _samples_360
    print(f"  LibriTTS total: {len(samples_libri):,} samples")
    CORPUS_REGISTRY["libri"] = samples_libri
else:
    print("  ⚠  LOAD_LIBRI=False — 'libri' not in registry.")

# ── SBCSAE ────────────────────────────────────────────────────────────────────
if LOAD_SBC:
    _all_sbc, _ = load_label_files_batched(BATCHED_ROOT_SBC, "sbcsae")
    # Split at load time: SBC001–005 → fixed eval holdout (never in registry)
    samples_sbc_test  = [s for s in _all_sbc
                         if s["sample_id"][:6] in SBC_TEST_IDS]
    samples_sbc_train = [s for s in _all_sbc
                         if s["sample_id"][:6] not in SBC_TEST_IDS]
    print(f"  SBC train: {len(samples_sbc_train):,}  |  SBC eval holdout: {len(samples_sbc_test):,}")
    CORPUS_REGISTRY["sbc"] = samples_sbc_train
else:
    print("  ⚠  LOAD_SBC=False — 'sbc' not in registry.")
    samples_sbc_test = []
    print("  ⚠  samples_sbc_test is empty — test set will be a random split.")

# ── BU Radio News — evaluation only, never in registry ────────────────────────
# Full BU corpus is used as the gold-standard break index evaluation set.
# BU is never added to CORPUS_REGISTRY or any training pool.
# Models trained without BU data can be published freely.
if LOAD_BU:
    samples_bu, _ = load_label_files_batched(BATCHED_ROOT_BU, "bu")
    print(f"  BU eval holdout: {len(samples_bu):,} samples (break index eval only — never in training)")
else:
    samples_bu = []
    print("  ⚠  LOAD_BU=False — BU break index evaluation will be skipped.")

# ── People's Speech ───────────────────────────────────────────────────────────
if LOAD_PS:
    samples_ps, _ = load_label_files_batched(BATCHED_ROOT_PS, "ps")
    CORPUS_REGISTRY["ps"] = samples_ps
else:
    print("  ⚠  LOAD_PS=False — 'ps' not in registry.")

# ── ADD FUTURE CORPORA HERE ───────────────────────────────────────────────────
# if LOAD_NEWCORPUS:
#     samples_newcorpus, _ = load_label_files_batched(BATCHED_ROOT_NEWCORPUS, "newcorpus")
#     CORPUS_REGISTRY["newcorpus"] = samples_newcorpus
# else:
#     print("  ⚠  LOAD_NEWCORPUS=False — 'newcorpus' not in registry.")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n✓ CORPUS_REGISTRY keys:", list(CORPUS_REGISTRY.keys()))
print(f"  SBC eval holdout (SBC001–005):  {len(samples_sbc_test):,} samples  [boundary + intonation]")
print(f"  BU eval holdout  (full corpus): {len(samples_bu):,} samples  [break index]")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4b — POS pre-generation & enrichment utilities         ║
# ║                                                              ║
# ║  Defines _generate_pos_files() and _load_pos_for_samples().  ║
# ║  These are called per-run inside run_experiment() when        ║
# ║  TRAIN_ON_POS=True for that run.                             ║
# ║                                                              ║
# ║  POS files are written once per corpus and reused across     ║
# ║  runs — generation is skipped for files that already exist.  ║
# ║  This cell is a no-op if TRAIN_ON_POS=False in Cell 1.       ║
# ╚══════════════════════════════════════════════════════════════╝

# ── spaCy load ────────────────────────────────────────────────────────────────
# Loaded unconditionally so it's available when any run needs it.
# en_core_web_sm was installed in Cell 2.
import spacy
from spacy.tokens import Doc
from tqdm.notebook import tqdm as tqdm_nb

_nlp_pos = spacy.load("en_core_web_sm", disable=["ner", "parser", "lemmatizer"])
print(f"✓ spaCy loaded: {_nlp_pos.pipe_names}")

def _generate_pos_files(batched_root):
    """
    For every batch_XXXX.json in batched_root, write a corresponding POS file
    to batched_root/pos/batch_XXXX.json. Skips files that already exist.

    Schema: { sample_id: [UPOS_string, ...] }  — one tag per word, word-level.

    Uses forced tokenization (Doc with words= and spaces=) over the full word
    list at once, matching the CLI inference pipeline. Runs tok2vec + tagger +
    attribute_ruler — attribute_ruler is what actually populates token.pos_
    (UPOS) from the tagger's raw TAG output; skipping it leaves pos_ empty
    on every token, which silently collapses to "X" everywhere.
    """
    pos_root = os.path.join(batched_root, "pos")
    os.makedirs(pos_root, exist_ok=True)

    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        print(f"  ⚠ No batch files in {batched_root} — nothing to tag.")
        return

    MAX_CONSECUTIVE_X = 5  # five X's in a row in one sample = tagging is broken

    n_new = 0
    for fname in tqdm_nb(batch_files, desc=f"POS tagging {os.path.basename(batched_root)}"):
        pos_path = os.path.join(pos_root, fname)
        if os.path.exists(pos_path):
            continue   # already done — skip

        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)

        pos_batch = {}
        for sid, data in batch.items():
            words = data["b"]["tokens"]
            if not words:
                pos_batch[sid] = []
                continue
            spaces = [True] * (len(words) - 1) + [False]
            doc = Doc(_nlp_pos.vocab, words=words, spaces=spaces)
            # attribute_ruler must run after tagger — see docstring above.
            for pipe_name in ("tok2vec", "tagger", "attribute_ruler"):
                if _nlp_pos.has_pipe(pipe_name):
                    _nlp_pos.get_pipe(pipe_name)(doc)
            tags = [t.pos_ or "X" for t in doc]

            # Sanity check: scan every sample for MAX_CONSECUTIVE_X X's in a
            # row. Catches both total-collapse failures (every tag is X) and
            # partial failures (e.g. 90% X with a few real tags scattered in)
            # that a "check only the first sample" approach would miss.
            run_length = 0
            for tag in tags:
                run_length = run_length + 1 if tag == "X" else 0
                if run_length >= MAX_CONSECUTIVE_X:
                    raise RuntimeError(
                        f"POS tagging sanity check failed for sample '{sid}' in "
                        f"{fname}: {MAX_CONSECUTIVE_X}+ consecutive 'X' tags found. "
                        f"This usually means a required spaCy pipe component was "
                        f"skipped. Aborting before writing more files — check "
                        f"_nlp_pos.pipe_names and the manual pipe-call loop above.\n"
                        f"  Sample tokens: {words}\n"
                        f"  Sample tags:   {tags}"
                    )

            pos_batch[sid] = tags

        with open(pos_path, "w") as f:
            json.dump(pos_batch, f)
        n_new += 1

    total = len(batch_files)
    print(f"  ✓ {n_new} new POS files written, {total - n_new} already existed "
          f"({total} total in {os.path.basename(batched_root)}).")


def _load_pos_for_samples(samples, batched_root):
    """
    Enrich each sample dict in-place with a 'pos_tags' key (list[str] of UPOS
    tags, word-level, same length as 'tokens').

    Reads from batched_root/pos/batch_XXXX.json. Falls back to ['X']*n for
    any sample not found in the POS files (safe fallback, logs a warning).

    Call _generate_pos_files(batched_root) first to ensure POS files exist.
    """
    pos_root = os.path.join(batched_root, "pos")
    pos_files = sorted(
        f for f in os.listdir(pos_root)
        if f.startswith("batch_") and f.endswith(".json")
    )

    pos_lookup = {}
    for fname in pos_files:
        with open(os.path.join(pos_root, fname)) as f:
            pos_lookup.update(json.load(f))

    n_missing = 0
    for s in samples:
        sid = s["sample_id"]
        if sid in pos_lookup:
            s["pos_tags"] = pos_lookup[sid]
        else:
            s["pos_tags"] = ["X"] * len(s["tokens"])
            n_missing += 1

    if n_missing:
        print(f"  ⚠ {n_missing} samples had no POS file entry — fell back to 'X'.")
    print(f"  ✓ POS tags loaded for {len(samples) - n_missing:,} / {len(samples):,} samples "
          f"from {os.path.basename(batched_root)}.")


print("✓ Cell 4b: POS utilities defined.")

---
## Section 5 · Dataset

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — ProsodyDataset + collate_fn                        ║
# ╚══════════════════════════════════════════════════════════════╝

PUNCT_SET = set(string.punctuation)

def is_punct_only(word):
    return bool(word) and all(c in PUNCT_SET for c in word)


class ProsodyDataset(Dataset):
    """
    Token-classification dataset for prosodic boundary detection.

    Label alignment rules
    ─────────────────────
    All three heads share the same rule: first sub-word of word i receives
    the word's label; continuation sub-words and special tokens get -100.

    Boundary   : 0 / 1  — all positions supervised
    Intonation : rising(1)→0 | falling(2)→1 | level(3)→2
                 none(0) and unclear(4) → -100  (masked)
    Break index: "3"→0 | "4"→1
                 "" and any missing value → -100 (masked)
                 ("" means unannotated, not an affirmative non-boundary)

    POS modes
    ─────────
    train_on_text=True,  train_on_pos=False : original text; no pos_ids tensor
    train_on_text=True,  train_on_pos=True  : original text + pos_ids tensor
                                              (post-transformer injection)
    train_on_text=False, train_on_pos=True  : POS abbreviation strings as input
                                              text; no pos_ids tensor needed
    """

    SPK_CHANGE_TOKEN = "/"

    # Universal Dependencies UPOS → short abbreviation (must match CLI inference.py)
    UNIVERSAL_TO_TOKEN = {
        "ADJ":   "adj", "ADP":   "adp", "ADV":   "adv", "AUX":   "aux",
        "CCONJ": "cc",  "DET":   "det", "INTJ":  "ij",  "NOUN":  "nn",
        "NUM":   "num", "PART":  "pt",  "PRON":  "pro", "PROPN": "np",
        "PUNCT": "pun", "SCONJ": "sc",  "SYM":   "sym", "VERB":  "vb",
        "X":     "xx",  "SPACE": "sp",
    }
    UNK_POS_TOKEN = "unk"

    def __init__(self, samples, tokenizer, max_length=128,
                 strip_punctuation=False,
                 train_on_text=True,
                 train_on_pos=False,
                 extra_features=None):
        self.tokenizer         = tokenizer
        self.max_length        = max_length
        self.strip_punctuation = strip_punctuation
        self.train_on_text     = train_on_text
        self.train_on_pos      = train_on_pos
        self.extra_features    = extra_features or []
        self.items             = [self._process(s) for s in samples]

    def _process(self, sample):
        words    = list(sample["tokens"])
        b_labels = list(sample["boundary_labels"])
        i_labels = list(sample.get("intonation_labels", []))
        x_labels = list(sample.get("break_idx_labels",  []))
        # Word-level UPOS tags. Present if sample was enriched by _load_pos_for_samples.
        # Defaults to "X" (unknown) if missing — safe fallback.
        pos_tags = list(sample.get("pos_tags", ["X"] * len(words)))

        if self.strip_punctuation:
            # POS tagging was done on the original word list before this step,
            # so tag quality is unaffected by STRIP_PUNCTUATION.
            quints = [
                (w, b, i, x, p)
                for w, b, i, x, p in zip(words, b_labels, i_labels, x_labels, pos_tags)
                if not is_punct_only(w) or w == self.SPK_CHANGE_TOKEN
            ]
            if quints:
                words, b_labels, i_labels, x_labels, pos_tags = map(list, zip(*quints))
            else:
                words, b_labels, i_labels, x_labels, pos_tags = [], [], [], [], []

        # ── Choose input sequence ─────────────────────────────────────────────
        # POS-only mode: replace each word with its POS abbreviation token.
        # DistilBERT reads e.g. "pro vb nn" instead of actual words.
        if self.train_on_text:
            input_words = words
        else:
            input_words = [
                self.UNIVERSAL_TO_TOKEN.get(p, self.UNK_POS_TOKEN)
                for p in pos_tags
            ]

        # ── Sub-word tokenization ─────────────────────────────────────────────
        encoding = self.tokenizer(
            input_words,
            is_split_into_words=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        word_ids = encoding.word_ids(batch_index=0)

        def _align(label_list, default=-100):
            aligned, prev = [], None
            for word_i in word_ids:
                if word_i is None:
                    aligned.append(default)
                elif word_i != prev:
                    aligned.append(label_list[word_i] if word_i < len(label_list) else default)
                else:
                    aligned.append(default)
                prev = word_i
            return aligned

        aligned_boundary = _align(b_labels)

        aligned_spk_mask = []
        prev = None
        for word_i in word_ids:
            if word_i is None:
                aligned_spk_mask.append(False)
            elif word_i != prev:
                aligned_spk_mask.append(words[word_i] == self.SPK_CHANGE_TOKEN)
            else:
                aligned_spk_mask.append(False)
            prev = word_i

        _INTONATION_MAP = {1: 0, 2: 1, 3: 2}
        aligned_intonation = []
        prev = None
        for word_i in word_ids:
            if word_i is None:
                aligned_intonation.append(-100)
            elif word_i != prev:
                raw = i_labels[word_i] if word_i < len(i_labels) else -1
                aligned_intonation.append(_INTONATION_MAP.get(raw, -100))
            else:
                aligned_intonation.append(-100)
            prev = word_i

        _BREAK_IDX_MAP = {"3": 0, "4": 1}
        aligned_break_idx = []
        prev = None
        for word_i in word_ids:
            if word_i is None:
                aligned_break_idx.append(-100)
            elif word_i != prev:
                raw = x_labels[word_i] if word_i < len(x_labels) else ""
                aligned_break_idx.append(_BREAK_IDX_MAP.get(raw, -100))
            else:
                aligned_break_idx.append(-100)
            prev = word_i

        # ── POS ID alignment (combined mode only) ─────────────────────────────
        # POS tag integer IDs aligned to sub-words. Only included in the item
        # when train_on_text=True and train_on_pos=True (combined mode).
        # PAD (0) at special tokens and continuation sub-words.
        item = {
            "sample_id":         sample["sample_id"],
            "tokens":            words,
            "input_ids":         encoding["input_ids"].squeeze(0),
            "attention_mask":    encoding["attention_mask"].squeeze(0),
            "spk_mask":          torch.tensor(aligned_spk_mask,   dtype=torch.bool),
            "labels":            torch.tensor(aligned_boundary,    dtype=torch.long),
            "intonation_labels": torch.tensor(aligned_intonation,  dtype=torch.long),
            "break_idx_labels":  torch.tensor(aligned_break_idx,   dtype=torch.long),
        }

        if self.train_on_text and self.train_on_pos:
            # Build POS tag vocabulary (must match CLI inference.py and Cell 4b)
            _pos_tag_names = ["PAD"] + list(self.UNIVERSAL_TO_TOKEN.keys())
            _pos_tag_to_id = {tag: i for i, tag in enumerate(_pos_tag_names)}
            aligned_pos = []
            prev = None
            for word_i in word_ids:
                if word_i is None:
                    aligned_pos.append(0)   # PAD for [CLS]/[SEP]
                elif word_i != prev:
                    tag = pos_tags[word_i] if word_i < len(pos_tags) else "X"
                    aligned_pos.append(_pos_tag_to_id.get(tag, _pos_tag_to_id.get("X", 0)))
                else:
                    aligned_pos.append(0)   # PAD for continuation sub-words
                prev = word_i
            item["pos_ids"] = torch.tensor(aligned_pos, dtype=torch.long)

        return item

    def __len__(self):  return len(self.items)
    def __getitem__(self, idx): return self.items[idx]


def prosody_collate_fn(batch):
    out = {
        "input_ids":         torch.stack([b["input_ids"]         for b in batch]),
        "attention_mask":    torch.stack([b["attention_mask"]    for b in batch]),
        "spk_mask":          torch.stack([b["spk_mask"]          for b in batch]),
        "labels":            torch.stack([b["labels"]            for b in batch]),
        "intonation_labels": torch.stack([b["intonation_labels"] for b in batch]),
        "break_idx_labels":  torch.stack([b["break_idx_labels"]  for b in batch]),
        "sample_ids":        [b["sample_id"] for b in batch],
        "tokens":            [b["tokens"]    for b in batch],
    }
    # pos_ids only present in combined mode — include if any item in the batch has it
    if "pos_ids" in batch[0]:
        out["pos_ids"] = torch.stack([b["pos_ids"] for b in batch])
    return out


print("✓ ProsodyDataset and prosody_collate_fn defined.")

---
## Section 7 · Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — ProsodyBoundaryModel                               ║
# ╚══════════════════════════════════════════════════════════════╝

# ── POS tag vocabulary (must match Cell 5 and CLI inference.py) ───────────────
_POS_TAG_NAMES = ["PAD"] + [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN",
    "NUM", "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB",
    "X", "SPACE",
]
POS_TAG_TO_ID = {tag: i for i, tag in enumerate(_POS_TAG_NAMES)}
NUM_POS_TAGS  = len(_POS_TAG_NAMES)   # 19

print(f"  POS vocabulary: {NUM_POS_TAGS} tags  (0=PAD, 1–{NUM_POS_TAGS-1}=UPOS)")


class ProsodyBoundaryModel(DistilBertPreTrainedModel):
    """
    Architecture
    ────────────
    DistilBERT encoder
        [+ optional POS embedding addition, post-transformer]
        └─► dropout (seq_classif_dropout)
             ├─► boundary_head    Linear(H → 2)   all positions
             ├─► intonation_head  Linear(H → 3)   rising / falling / level
             └─► break_idx_head   Linear(H → 2)   index-3 / index-4

    POS embedding design (combined mode)
    ─────────────────────────────────────
    When use_pos_embedding=True, a small nn.Embedding(NUM_POS_TAGS, pos_emb_dim)
    maps integer POS IDs to vectors of size pos_emb_dim (default 64). A Linear
    projection then maps these to H=768, and the result is ADDED to DistilBERT's
    last hidden state AFTER the transformer.

    Post-transformer injection leaves DistilBERT's pretrained weights fully
    intact and makes the POS contribution easily ablatable.
    """

    def __init__(self, config):
        super().__init__(config)
        self.distilbert = DistilBertModel(config)
        self.dropout    = nn.Dropout(config.seq_classif_dropout)

        # ── Optional POS embedding (combined mode only) ───────────────────
        self.use_pos_embedding = getattr(config, "use_pos_embedding", False)
        if self.use_pos_embedding:
            _pos_emb_dim  = getattr(config, "pos_emb_dim",  64)
            _num_pos_tags = getattr(config, "num_pos_tags", NUM_POS_TAGS)
            self.pos_embedding = nn.Embedding(
                _num_pos_tags, _pos_emb_dim, padding_idx=0
            )
            self.pos_proj = nn.Linear(_pos_emb_dim, config.hidden_size, bias=False)

        # ── Classification heads ──────────────────────────────────────────
        self.boundary_head   = nn.Linear(config.hidden_size, 2)
        self.intonation_head = nn.Linear(config.hidden_size, 3)
        self.break_idx_head  = nn.Linear(config.hidden_size, 2)
        self.post_init()

    def forward(self, input_ids, attention_mask, pos_ids=None,
                labels=None, intonation_labels=None, break_idx_labels=None,
                loss_weights=None):
        outputs = self.distilbert(input_ids=input_ids,
                                  attention_mask=attention_mask)
        seq_out = self.dropout(outputs.last_hidden_state)   # (B, T, H)

        # POS injection: only in combined mode (use_pos_embedding=True + pos_ids provided)
        if self.use_pos_embedding and pos_ids is not None:
            pos_emb = self.pos_proj(self.pos_embedding(pos_ids))   # (B, T, H)
            seq_out = seq_out + pos_emb

        boundary_logits   = self.boundary_head(seq_out)    # (B, T, 2)
        intonation_logits = self.intonation_head(seq_out)  # (B, T, 3)
        break_idx_logits  = self.break_idx_head(seq_out)   # (B, T, 2)

        losses = {}
        if labels is not None:
            losses["boundary"] = nn.CrossEntropyLoss(
                weight=loss_weights, ignore_index=-100)(
                boundary_logits.view(-1, 2), labels.view(-1))
        if intonation_labels is not None and (intonation_labels != -100).any():
            losses["intonation"] = nn.CrossEntropyLoss(ignore_index=-100)(
                intonation_logits.view(-1, 3), intonation_labels.view(-1))
        if break_idx_labels is not None and (break_idx_labels != -100).any():
            losses["break_idx"] = nn.CrossEntropyLoss(ignore_index=-100)(
                break_idx_logits.view(-1, 2), break_idx_labels.view(-1))

        out = {
            "boundary_logits":   boundary_logits,
            "intonation_logits": intonation_logits,
            "break_idx_logits":  break_idx_logits,
        }
        if losses:
            out["losses"] = losses
        return out

    @classmethod
    def _can_set_experts_implementation(cls):
        return False


print("✓ ProsodyBoundaryModel defined.")
print(f"  Heads: boundary(2)  intonation(3)  break_idx(2)")
print(f"  POS embedding: controlled per-run via config.use_pos_embedding")

---
## Section 8 · Metrics

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Metrics helpers                                    ║
# ╚══════════════════════════════════════════════════════════════╝

def flatten_predictions(all_logits, all_labels, all_spk_masks=None):
    flat_preds, flat_labels = [], []
    for idx, (logits_batch, labels_batch) in enumerate(zip(all_logits, all_labels)):
        preds_batch = logits_batch.argmax(dim=-1)
        spk_batch   = all_spk_masks[idx] if all_spk_masks else None
        for b_i, (preds_seq, labels_seq) in enumerate(zip(preds_batch, labels_batch)):
            mask = labels_seq != -100
            if spk_batch is not None:
                mask = mask & ~spk_batch[b_i].to(mask.device)
            flat_preds.extend(preds_seq[mask].cpu().tolist())
            flat_labels.extend(labels_seq[mask].cpu().tolist())
    return np.array(flat_preds), np.array(flat_labels)

def compute_metrics(flat_preds, flat_labels):
    return {
        "f1":        f1_score(flat_labels,        flat_preds, pos_label=1, zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, pos_label=1, zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, pos_label=1, zero_division=0),
    }

def compute_multiclass_metrics(flat_preds, flat_labels):
    return {
        "f1":        f1_score(flat_labels,        flat_preds, average="macro", zero_division=0),
        "precision": precision_score(flat_labels, flat_preds, average="macro", zero_division=0),
        "recall":    recall_score(flat_labels,    flat_preds, average="macro", zero_division=0),
    }

print("✓ Metrics helpers defined.")
